# Data ingestion

This notebook is to ingest raw data to the MOTEL platform. In this step, the user/contributor can ingest the data in a flexible structure with certain required information (defined in the schema for 'unmapped_entity').

The aim of data ingestion process is to provide a standard (i.e. schema of `unmapped entity`) to ingest data into the MOTEL platform.

## How to ingest your data here?
1. Understand the guideline and understand the required structure/format (schema) for the unmapped entity for MOTEL platform.
2. Convert your data into the `unmapped entity` based on the information in the previous step. An Example is provided in the folder `ingestion_space`.
3. Run the validation to make sure the unmapped entity you've created follow the required structure/format (i.e. schema).


This notebook is divided into:
1. What are the format and required information of the data to be ingested (also calle `unmapped entity`)?
2. Run the validation

# Step 1: Understand required structure/format (schema) for `unmapped_entity`

In [2]:
from pathlib import Path
import yaml

# Find schema_simple whether the kernel starts in 1_ingest/ or the repository root.
schema_candidates = [Path('../schema_simple'), Path('schema_simple')]
SCHEMA_DIR = next((path.resolve() for path in schema_candidates if path.is_dir()), None)
if SCHEMA_DIR is None:
    raise FileNotFoundError(
        "Could not find schema_simple. Start the notebook from 1_ingest/ "
        "or from the repository root."
    )

# Load every YAML schema recursively, preserving its path below schema_simple.
schema_files = sorted(SCHEMA_DIR.rglob('*.yaml'))
schemas = {}
for schema_path in schema_files:
    relative_name = schema_path.relative_to(SCHEMA_DIR).as_posix()
    with open(schema_path, 'r', encoding='utf-8') as f:
        schemas[relative_name] = yaml.safe_load(f)

schema_unmapped = schemas['unmapped_entity.yaml']
print(f'Loaded {len(schemas)} schemas from {SCHEMA_DIR}')

Loaded 13 schemas from E:\Barton\repositories\motel-platform\schema_simple


In [3]:
def print_tree(node, name="root", prefix=""):
    # Print root
    if prefix == "":
        print(name)

    if isinstance(node, dict):
        items = list(node.items())
        for i, (key, value) in enumerate(items):
            is_last = i == len(items) - 1
            branch = "└── " if is_last else "├── "
            next_prefix = prefix + ("    " if is_last else "│   ")

            if isinstance(value, dict):
                print(f"{prefix}{branch}{key}")
                print_tree(value, prefix=next_prefix)
            elif isinstance(value, list):
                print(f"{prefix}{branch}{key}: {', '.join(map(str, value))}")
            else:
                print(f"{prefix}{branch}{key}: {value}")

    elif isinstance(node, list):
        for i, value in enumerate(node):
            is_last = i == len(node) - 1
            branch = "└── " if is_last else "├── "
            next_prefix = prefix + ("    " if is_last else "│   ")

            label = f"[{i}]"
            if isinstance(value, (dict, list)):
                print(f"{prefix}{branch}{label}")
                print_tree(value, prefix=next_prefix)
            else:
                print(f"{prefix}{branch}{label}: {value}")

## Unmapped entity schema

This is the ingestion contract loaded from `schema_simple/unmapped_entity.yaml`.

In [4]:
# Show the unmapped entity schema as a tree.
print_tree(schema_unmapped, name="unmapped_entity.yaml")

unmapped_entity.yaml
└── UnmappedEntity
    ├── technology_name: [Required; String] The literal unparsed common name or label of the technology as specified in the raw source material.
    ├── technology
    │   ├── technology_description: [String] Free-text engineering parameters describing the technical baseline of the asset.
    │   ├── technology_type: [String] The structural archetype classifying how the physical asset interacts with energy streams. Suggested: ['conversion', 'storage', 'distribution', 'consumption']
    │   ├── technology_category: [String] Broad domain or energy paradigm grouping for the hardware. Suggested: ['renewable', 'fossil_fuel', 'nuclear', 'synthetic']
    │   ├── technology_assumption: [String] Explicit technical constraints, design variations, or baseline modifications assumed by the raw source provider.
    │   ├── process_name: [String] The raw action-oriented label identifying the conversion or functional service taking place.
    │   ├── process_typ

## Available simple schemas

`schema_simple` contains separate YAML files for linked entities, controlled vocabularies, secondary entities, and supplementary records.

In [5]:
# List all schemas discovered recursively under schema_simple.
for relative_name, schema in schemas.items():
    title = schema.get('title', '') if isinstance(schema, dict) else ''
    print(f'- {relative_name}' + (f'  ({title})' if title else ''))

- controlled_vocabulary/attribute.yaml
- controlled_vocabulary/capacity_scope.yaml
- controlled_vocabulary/carrier.yaml
- controlled_vocabulary/geographic_scope.yaml
- controlled_vocabulary/system_boundary.yaml
- controlled_vocabulary/temporal_scope.yaml
- linked_entity.yaml
- secondary/process.yaml
- secondary/source.yaml
- secondary/technology.yaml
- supplementary/contributor.yaml
- supplementary/review.yaml
- unmapped_entity.yaml
